In [12]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.agents import initialize_agent, Tool
from langchain.chains import LLMChain, RetrievalQA
from langchain.vectorstores import FAISS
from langchain.prompts import PromptTemplate
import os
from langchain_openai import AzureChatOpenAI


In [2]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [5]:
reports_and_outlooks_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_0.faiss", embeddings).as_retriever(search_kwargs={"k":5})
finanacial_and_credit_info_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_1.faiss", embeddings).as_retriever(search_kwargs={"k":5})
presentations_and_infographic_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_2.faiss", embeddings).as_retriever(search_kwargs={"k":5})
corporate_documents_retriever=FAISS.load_local(r"C:\Users\raju220738\Downloads\vectorstore_3.faiss", embeddings).as_retriever(search_kwargs={"k":5})

In [6]:
tools=[ Tool(
    name="Reports and Outlooks",
    func=reports_and_outlooks_retriever.get_relevant_documents,
    description="Retrieves relevant reports and outlooks related documents"
),
Tool(
    name="Financial and Credit Info",
    func=finanacial_and_credit_info_retriever.get_relevant_documents,
    description="Retrieves relevant financial and credit info related documents"
),
Tool(
    name="Presentations and Infographic",
    func=presentations_and_infographic_retriever.get_relevant_documents,
    description="Retrieves relevant presentations and infographic related documents"
),
Tool(
    name="Corporate Documents",
    func=corporate_documents_retriever.get_relevant_documents,
    description="Retrieves relevant corporate documents related documents"
)]

In [13]:
# Using Azure OpenAI API for embeddings/llms
os.environ["AZURE_OPENAI_API_KEY"] = "95d5f4e4a96a46a4ba1900155e8b641c"

llm = AzureChatOpenAI(
    azure_endpoint="https://creditmemo.openai.azure.com/",
    azure_deployment='gpt-4o',
    openai_api_version='2024-02-15-preview')

In [14]:
agent=initialize_agent(tools,llm,agent="zero-shot-react-description",handle_parsing=True,verbose=True)

C:\Users\raju220738\AppData\Local\Temp\1\ipykernel_18548\638906484.py:1: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent=initialize_agent(tools,llm,agent="zero-shot-react-description",handle_parsing=True,verbose=True)


In [15]:
decomposition_prompt=PromptTemplate.from_template(
    "Break the complex query '{query}' into more than 3 focused sub-questions, without extra context or headings."
)
decomposition_chain=LLMChain(llm=llm,prompt=decomposition_prompt)
response_merging_prompt=PromptTemplate.from_template(
    "Here are partial answers to a complex query:\n\n{sub_answer}\n\n"
    "Based on these partial answers, genrate a single , structured final response."
)

C:\Users\raju220738\AppData\Local\Temp\1\ipykernel_18548\1288142599.py:4: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  decomposition_chain=LLMChain(llm=llm,prompt=decomposition_prompt)


In [16]:
response_merging_chain=LLMChain(llm=llm,prompt=response_merging_prompt)

In [17]:
def process_complex_query(query):
    sub_questions=decomposition_chain.run(query).split("\n")
    sub_questions=[sub_question.strip() for sub_question in sub_questions if sub_question.strip()]
    print("\n Generated sub-questions")
    for i, sub_question in enumerate(sub_questions,1):
        print(f"{i}. {sub_question}")
    sub_answers={sub_q: agent.run(sub_q) for sub_q in sub_questions}
    sub_answers_str = [f"{sub_question}: {sub_answers}" for sub_question in sub_questions]
    final_response = response_merging_chain.run("\n".join(sub_answers_str))
    return final_response

In [18]:
query=["How was the financial performance of Verizon in 2023 compared to performance in 2022?"]
response=process_complex_query(query)

C:\Users\raju220738\AppData\Local\Temp\1\ipykernel_18548\3440789730.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  sub_questions=decomposition_chain.run(query).split("\n")



 Generated sub-questions
1. What was Verizon's revenue in 2023?
2. What was Verizon's revenue in 2022?
3. What were Verizon's net profits in 2023?
4. What were Verizon's net profits in 2022?
5. How did Verizon's operating expenses in 2023 compare to those in 2022?
6. What was the growth rate in Verizon's key financial metrics between 2022 and 2023?
7. How did Verizon's stock price perform in 2023 compared to 2022?


> Entering new AgentExecutor chain...
To find Verizon's revenue in 2023, I should look for financial documents or reports that contain revenue information.

Action: Financial and Credit Info
Action Input: "Verizon revenue 2023"
Observation: [Document(metadata={'Title': '{\'text\': \'"Fitch Affirms Verizon\\\'s \\\'A-\\\' Rating; Highlights Market Leadership and Spectrum Acquisition"\'}', 'keywords': '{\'text\': "1. Ratings impact\\n2. Gross debt\\n3. Credit facilities\\n4. Key rating drivers\\n5. Wireless market leader\\n6. Verizon\\n7. Market share leadership\\n8. Wireles

Thought:The documents retrieved did not provide specific revenue figures for Verizon in 2023. To obtain this information, I need to look for reports or presentations that might include financial results or summaries.

Action: Reports and Outlooks
Action Input: "Verizon 2023 revenue"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon 2023 Annual Report: Revenue Shifts and Service Growth"\'}', 'keywords': "{'text': '1. Consumer total operating revenues\\n2. Wireless equipment revenue decrease\\n3. Other revenue decrease\\n4. Service revenue increase\\n5. Wireless service revenue growth\\n6. Access revenues increase\\n7. Administrative and telco recovery charges\\n8. FWA subscriber base increase\\n9. Device protection revenue increase\\n10. Wireless equipment sales promotions amortization\\n11. Non-retail service revenue growth\\n12. TravelPass revenue increase\\n13. Prepaid subscriber base decrease\\n14. Fios service revenue\\n15. Wireless device sales volume decrease\\n16.

Thought:The retrieved document, "Verizon 2023 Annual Financial Overview: Consolidated Income and Comprehensive Statements," provides the total operating revenues for Verizon in 2023. 

Final Answer: Verizon's total operating revenues in 2023 were $133.974 billion.

> Finished chain.


> Entering new AgentExecutor chain...
To find out Verizon's revenue in 2022, I should look for financial and credit information related to Verizon. This will likely contain the revenue figures for that year.

Action: Financial and Credit Info
Action Input: "Verizon 2022 revenue"
Observation: [Document(metadata={'Title': '{\'text\': \'"Fitch Affirms Verizon\\\'s \\\'A-\\\' Rating; Highlights Market Leadership and Spectrum Acquisition"\'}', 'keywords': '{\'text\': "1. Ratings impact\\n2. Gross debt\\n3. Credit facilities\\n4. Key rating drivers\\n5. Wireless market leader\\n6. Verizon\\n7. Market share leadership\\n8. Wireless subscriber base\\n9. U.S. carriers\\n10. Postpaid customers\\n11. Competitive wir

Thought:The financial documents retrieved did not contain the specific revenue figure for Verizon in 2022. Therefore, I should try another source, possibly presentations or corporate documents, that might provide an annual report or financial summary including the revenue details.

Action: Presentations and Infographic
Action Input: "Verizon 2022 revenue"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon\\\'s Q3 2024: Growth in Wireless and Broadband Amid Stable Revenue"\'}', 'keywords': "{'text': '1. Verizon\\n2. 3Q 2024 Earnings\\n3. Wireless customer growth\\n4. Broadband customer growth\\n5. Consolidated cash flow\\n6. Total consolidated revenue\\n7. Cash flow from operations\\n8. Capital expenditures\\n9. Free cash flow\\n10. Dividends paid\\n11. Adjusted EBITDA\\n12. Adjusted EPS\\n13. 3Q Revenue\\n14. Wireless service revenue\\n15. Y/Y growth\\n16. Total consumer revenue\\n17. Total business revenue\\n18. Broadband net additions\\n19. Fios internet net additions\\

...

Thought:The documents provided primarily focus on Verizon's credit rating, debt, and financial outlook rather than specific year-over-year growth in key financial metrics between 2022 and 2023. They mention stable margins, revenue growth expectations, and capital spending related to 5G, but do not provide a detailed account of financial growth metrics for the years in question.

To accurately determine the growth rate in Verizon's key financial metrics between 2022 and 2023, I would need more specific data or reports focused on those years' financial performance, such as income statements or detailed financial reports.

Since the available documents did not provide a direct answer, I will attempt to find more specific financial data.

Action: Reports and Outlooks
Action Input: "Verizon financial growth 2022 2023"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon 2023 Financial Overview: Operating Cash Increase Amid Investment and Financing Reductions"\'}', 'keywords': "{

Thought:The documents provide information on Verizon's financial activities and performance during 2022 and 2023. Here's a summary of the key financial metrics growth between these years:

1. **Operating Cash Flow**: Increased by $334 million from $37,141 million in 2022 to $37,475 million in 2023. This increase was primarily due to improvements in working capital.

2. **Investing Activities**: Cash used in investing activities decreased from $28,662 million in 2022 to $23,432 million in 2023. This was mainly due to a decrease in capital expenditures, which decreased by $4.3 billion due to the completion of the C-Band deployment program.

3. **Financing Activities**: Cash used in financing activities increased from $8,529 million in 2022 to $14,657 million in 2023. The increase was attributed to various financing decisions made by the company.

4. **Capital Expenditures**: Decreased from $23.1 billion in 2022 to $18.8 billion in 2023.

5. **Net Income**: Decreased from $21,256 million 

Thought:The search did not return specific information about Verizon's stock price performance in 2023 compared to 2022. I should look into other types of documents that might contain this data, such as presentations or corporate documents. 

Action: Presentations and Infographic
Action Input: "Verizon stock price performance 2023 compared to 2022"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon 3Q24 Financial Highlights: Mobility Gains, Broadband Growth, Strategic Execution"\'}', 'keywords': "{'text': '1. SEC rules  \\n2. Non-GAAP financial measures  \\n3. GAAP measures  \\n4. Reconciliation  \\n5. Presentation  \\n6. Verizon  \\n7. Investors  \\n8. Continued momentum  \\n9. 3Q24  \\n10. Mobility improvement  \\n11. Operational performance  \\n12. Financial performance  \\n13. Postpaid phone net adds  \\n14. Year-over-Year (YoY)  \\n15. Prepaid net adds  \\n16. SafeLink  \\n17. Broadband growth  \\n18. FWA subscribers  \\n19. Strategic moves  \\n20. Cost transformatio

Thought:The search did not return specific information about Verizon's stock price performance in 2023 compared to 2022. I should look into corporate documents that might contain this data, such as annual reports or shareholder communications.

Action: Corporate Documents
Action Input: "Verizon stock price performance 2023 compared to 2022"
Observation: [Document(metadata={'Title': '{\'text\': \'"Verizon 2023 ESG Report Summary"\'}', 'keywords': "{'text': '1. Verizon  \\n2. 2023  \\n3. ESG  \\n4. Report  \\n5. Year-end  \\n6. Endnotes'}"}, page_content='*As of year-end 2023 unless otherwise noted.\n\nVerizon 2023 ESG Report\n# Verizon 2023 ESG Report\n\n# Endnotes'), Document(metadata={'Title': '{\'text\': \'"Verizon 2023 ESG Report: Non-Financial Disclosures and Performance Overview"\'}', 'keywords': "{'text': '1. Verizon  \\n2. ESG Report  \\n3. 2023  \\n4. Non-financial disclosures  \\n5. Financial disclosures  \\n6. Annual Report  \\n7. Form 10-K  \\n8. Verizon operations  \\n9. Tr

Thought:The search did not provide specific details about Verizon's stock price performance in 2023 compared to 2022. None of the documents retrieved contained direct information on Verizon's stock prices for these years.

Final Answer: I was unable to find specific information about Verizon's stock price performance in 2023 compared to 2022 from the available documents. You may want to check financial news websites or stock market analysis platforms for detailed stock price information.

> Finished chain.


In [19]:
print(response)

Certainly! Here is a structured final response based on the partial answers provided:

---

### Verizon Financial Overview: 2022 vs. 2023

#### Revenue:
- **2023**: Verizon's total operating revenues were $133.974 billion.
- **2022**: The specific revenue figure for 2022 was not retrieved from the available documents. For precise financial details, consulting Verizon's official annual report or Form 10-K filing would be necessary.

#### Net Profits:
- **2023**: The specific net profits of Verizon for 2023 are not available in the documents retrieved. For precise figures, please refer to Verizon's 2023 Annual Report or Form 10-K filings.
- **2022**: Similarly, the specific net profit figure for 2022 is not available from the accessed documents. This information is typically found in the company's annual report or Form 10-K filed with the SEC.

#### Operating Expenses:
- Verizon's operating expenses in 2023 were $111,097 million, compared to $106,368 million in 2022, marking an increase 